# 레슨 01 — 미션 정답지

> 🔒 **교사·관리자 전용. 학생에게 배포 금지.**
> 학생에게는 `mission.ipynb` 만 공유한다. 이 파일은 채점·강의 준비용이다.

각 미션의 모범 답안, 기대 출력, 자주 보이는 오답 패턴, 채점 포인트를 정리했다.

## 0. 환경 셀

In [ ]:
import os
import numpy as np

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ
DATA_BASE = "./data"

print("numpy:", np.__version__, "| data base:", DATA_BASE)

---

## 미션 1 정답

In [ ]:
scores = np.loadtxt(f"{DATA_BASE}/scores.csv", delimiter=",", skiprows=1)

n = scores.size
mean = scores.mean()
median = np.median(scores)
std = scores.std()
high = int(scores.max())
low = int(scores.min())
n_high = scores[scores >= 90].size
n_low = scores[scores < 60].size

print(f"학생 수: {n}명")
print(f"평균: {mean:.2f}점, 중앙값: {median:.1f}점")
print(f"표준편차: {std:.2f}")
print(f"최고/최저: {high} / {low}")
print(f"90점 이상: {n_high}명, 60점 미만: {n_low}명")

### 기대 출력 (실제 데이터 기준 — 강사 수업 전 확인 필수)

```
학생 수: 200명
평균: 75.07점, 중앙값: 76.0점
표준편차: 11.27
최고/최저: 100 / 35
90점 이상: 17명, 60점 미만: 21명
```

### 채점 포인트

- `skiprows=1` 빠뜨려서 헤더가 float 변환 실패하는 학생이 가장 많음. 에러 메시지를 같이 읽으며 깨닫게 함.
- 평균 / 표준편차 소수점 표기 안 맞아도 통과. 값만 정확하면 OK.
- `ddof=0` 가 NumPy 기본이라 학생 답이 11.27 로 나옴. `ddof=1` 로 풀면 11.30 부근이 나오는데 둘 다 통과 처리.
- `len(scores[mask])` 또는 `mask.sum()` 도 정답.

### 자주 보이는 오답

In [ ]:
# 오답: 90점 이상이 0명으로 나옴
scores = np.loadtxt(f"{DATA_BASE}/scores.csv", delimiter=",")  # skiprows 없음
# → 헤더 "score" 가 float 으로 못 바뀌어 ValueError. 학생이 첫 줄 직접 지운 다음 진행하면 위 출력은 나옴.

---

## 미션 2 정답

In [ ]:
raw = np.loadtxt(f"{DATA_BASE}/daily_steps.csv", delimiter=",", skiprows=1)
day = raw[:, 0].astype(int)
steps = raw[:, 1]

total = int(steps.sum())
avg = steps.mean()
std = steps.std()

mask_10k = steps >= 10000
mask_low = steps < 5000
days_low = day[mask_low].tolist()
rate_10k = mask_10k.mean() * 100

i_max, i_min = steps.argmax(), steps.argmin()

print(f"총 걸음: {total:,}")
print(f"일평균: {avg:,.0f} 보 (표준편차 {std:,.0f})")
print(f"10,000보 달성: {mask_10k.sum()}일 ({rate_10k:.1f}%)")
print(f"5,000보 미만: {mask_low.sum()}일, 일자: {days_low}")
print(f"최대일: {day[i_max]}일차, {int(steps[i_max]):,}보")
print(f"최소일: {day[i_min]}일차, {int(steps[i_min]):,}보")

### 기대 출력 (값은 데이터 갱신마다 다시 확인)

```
총 걸음: 723,xxx (구체 값 강사 사전 확인)
일평균: 8,xxx 보 (표준편차 2,xxx)
10,000보 달성: 약 18~22일 (20~24%)
5,000보 미만: 약 18~22일
최대일: <약 11,500보 이상의 어느 평일>
최소일: <약 2,100보의 82일차 또는 그 부근>
```

### 채점 포인트

- 일자 리스트가 비었으면 boolean indexing 실수.
- `argmax` 위치를 그대로 출력값으로 쓰면 안 됨(인덱스 vs 실제 day 값 구분).
- `mask.mean() * 100` 으로 달성률 쉽게 구할 수 있음. 학생이 `n_high / size * 100` 처럼 풀어도 정답.

### 자주 보이는 오답

In [ ]:
# 오답: 일자 대신 인덱스를 출력
print(f"최대일: {i_max}일차, {steps[i_max]}보")
# → 인덱스는 0부터, day 는 1부터. 학생에게 둘의 차이를 강조.

---

## 미션 3 정답

In [ ]:
raw = np.loadtxt(f"{DATA_BASE}/temperatures.csv", delimiter=",", skiprows=1)
day = raw[:, 0].astype(int)
temp = raw[:, 1]

mean = temp.mean()
median = np.median(temp)
std = temp.std()
q25, q50, q75 = np.quantile(temp, [0.25, 0.50, 0.75])
iqr = q75 - q25

below_zero_mask = temp < 0
below_zero_days = day[below_zero_mask].tolist()

hot_mask = temp > q75
hot_days = day[hot_mask].tolist()

mild_mask = (temp >= 10) & (temp <= 15)

print(f"평균: {mean:.2f}, 중앙값: {median:.2f}, 표준편차: {std:.2f}")
print(f"Q1: {q25:.2f}, Q2: {q50:.2f}, Q3: {q75:.2f}, IQR: {iqr:.2f}")
print(f"영하 일수: {len(below_zero_days)}일, 일자: {below_zero_days}")
print(f"Q3 초과 일수: {len(hot_days)}일, 일자: {hot_days}")
print(f"10~15°C 일수: {mild_mask.sum()}일")

### 기대 출력 (대략)

```
평균: 약 8~10°C, 중앙값: 약 8~9°C
Q1: 약 4, Q2: 약 8~9, Q3: 약 13~14, IQR: 약 9
영하: 2~3일 (8, 9일차의 콜드스냅 부근)
Q3 초과: 약 22일
10~15°C 일수: 약 20~25일
```

### 채점 포인트

- `>` 와 `>=` 의 차이를 의식했는지가 핵심. 이번 기준은 "Q3 보다 높은(strictly greater)" 이므로 `>` 가 맞음.
- and 조건 괄호 빠뜨리면 결과가 이상해진다. `temp >= 10 & temp <= 15` 는 우선순위 때문에 의도와 다름.

---

## 미션 4 정답

In [ ]:
prices = np.loadtxt(f"{DATA_BASE}/product_prices.csv", delimiter=",", skiprows=1)

n = prices.size
mean = prices.mean()
median = np.median(prices)
std = prices.std()
high = int(prices.max())
low = int(prices.min())
above_mean = prices[prices > mean].size
above_median = prices[prices > median].size

print(f"상품 수: {n}")
print(f"평균: {mean:,.0f}원, 중앙값: {median:,.0f}원, 표준편차: {std:,.0f}원")
print(f"최고: {high:,}원, 최저: {low:,}원")
print(f"평균 초과: {above_mean}개")
print(f"중앙값 초과: {above_median}개")

### 기대 출력 (대략)

```
상품 수: 100
평균: 약 24,000원, 중앙값: 약 20,000원, 표준편차: 약 15,000원
최고: 약 86,000원, 최저: 약 7,600원
평균 초과: 약 35~40개
중앙값 초과: 50개 (정의상)
```

### 한 줄 결론 채점

**통과**: 평균과 중앙값의 차이를 해석한 문장. 데이터 근거 1개 이상.
> "평균 24,000원이 중앙값 20,000원보다 4,000원 높다. 4만원 이상의 비싼 상품이 평균을 끌어올린 결과다."

**미통과**: 숫자만 다시 말한 문장.
> "평균은 24,000원이고 중앙값은 20,000원이다."

**잘된 보너스 표현**: "보통 사람이 보는 가격" 이라는 표현을 중앙값에 붙이는 학생.

---

## 미션 5 정답

In [ ]:
mask_a = scores >= 90
mask_b = (scores >= 80) & (scores < 90)
mask_c = (scores >= 70) & (scores < 80)
mask_d = (scores >= 60) & (scores < 70)
mask_f = scores < 60

a, b, c, d, f = mask_a.sum(), mask_b.sum(), mask_c.sum(), mask_d.sum(), mask_f.sum()

print(f"A (90+ ): {a}명")
print(f"B (80-89): {b}명")
print(f"C (70-79): {c}명")
print(f"D (60-69): {d}명")
print(f"F (<60 ): {f}명")
print(f"합계: {a + b + c + d + f}명")
assert a + b + c + d + f == scores.size

### 기대 출력 (대략)

```
A: 17, B: 약 50~55, C: 약 60~65, D: 약 35~40, F: 21
합계: 200
```

### 채점 포인트

- 5개 마스크가 서로 겹치지 않고 합이 정확히 200 이어야 통과.
- 학생이 `>` 와 `>=` 경계를 잘못 써서 합이 199 또는 201 이 나오는 경우 자주 발생. 합계 검증 셀이 이걸 잡는다.

### 자주 보이는 오답

In [ ]:
mask_b = (scores >= 80) & (scores <= 90)   # 90 도 포함시킴 → A 와 중복
# 합이 200 + a 명만큼 더 나옴

---

## 보너스 답안 (참고용)

### B1 — ASCII 막대 그래프

In [ ]:
labels = [("A", a), ("B", b), ("C", c), ("D", d), ("F", f)]
for label, count in labels:
    bar = "█" * (count // 2)  # 2명당 1칸
    print(f"{label}: {bar} {count}")

### B2 — 평일/주말 분리

In [ ]:
dow = (day - 1) % 7
weekend_mask = (dow == 5) | (dow == 6)
print(f"평일 평균: {steps[~weekend_mask].mean():.0f}보")
print(f"주말 평균: {steps[weekend_mask].mean():.0f}보")

### B3 — 상위 3명 인덱스

In [ ]:
above_mean_idx = np.where(scores > scores.mean())[0]
top3 = above_mean_idx[np.argsort(scores[above_mean_idx])[-3:]]
print("상위 3명 인덱스:", top3.tolist(), "점수:", scores[top3].tolist())

---

## 학생이 자주 실수하는 부분 (전체)

| 실수 | 원인 | 지도 방법 |
|---|---|---|
| `np.loadtxt` 에서 `delimiter` 안 줌 | 기본 구분자가 공백 | 첫 줄에 `delimiter=","` 강조 |
| `skiprows=1` 빠뜨림 | CSV 헤더가 문자라 에러 | 에러 메시지 "could not convert string to float" 를 보면 떠올리도록 |
| `scores[scores>=90].size` 대신 `len(scores[scores>=90])` 씀 | 둘 다 작동 | 둘 다 정답 처리하되 `.size` 추천 |
| `>=` 와 `>` 혼동 | 경계 처리 헷갈림 | 미션 5 합계 검증으로 자연스럽게 노출 |
| `and`/`or` 를 boolean array 에 사용 | 비트 연산자 모름 | `&`, `|` + 괄호 |
| 평균/중앙값 차이 못 설명 | 개념 부족 | 미션 4 결론에서 의도적으로 노출 |